In [1]:
from scripts.data_collector.us_index.collector import SP500Index, SP400Index
from scripts.yfinance_collector import collect

import os
import sys
from pathlib import Path

import pandas as pd
qlib_dir = Path().resolve().joinpath("qlib_data_US")
print (qlib_dir)

D:\Projects\Investing\QuantitativeML_Investing\Database\qlib_data_US


## Get the list of all tickers in SP500 index

In [12]:
SP500 = SP500Index(
    index_name="SP500",
    qlib_dir=qlib_dir
)
SP500.save_new_companies()

2024-05-15 12:02:57.519 | INFO     | scripts.data_collector.us_index.collector:get_new_companies:129 - get new companies SP500 ......
2024-05-15 12:02:58.223 | INFO     | scripts.data_collector.us_index.collector:get_new_companies:137 - end of get new companies SP500 ......


In [3]:
tickers_sp500 = pd.read_csv(qlib_dir.joinpath("instruments/sp500_only_new.txt"), sep='\t', header=None, names=['symbol', 'start_date', 'end_date'])
tickers_sp500

,symbol,start_date,end_date
0,MMM,1999-01-01,2099-12-31
1,AOS,1999-01-01,2099-12-31
2,ABT,1999-01-01,2099-12-31
3,ABBV,1999-01-01,2099-12-31
4,ACN,1999-01-01,2099-12-31
...,...,...,...
498,XYL,1999-01-01,2099-12-31
499,YUM,1999-01-01,2099-12-31
500,ZBRA,1999-01-01,2099-12-31
501,ZBH,1999-01-01,2099-12-31


## Collect data from yahoo finance to the data dock

In [15]:
tickers = (
    ['^GSPC']  # '^MID', '^SP600'
    + tickers_sp500['symbol'].tolist()
)
tickers = [ticker.replace('.', '-') for ticker in tickers]


collect(
    tickers,
    qlib_dir,
    start="2000-01-01", end="2030-01-01",
    append_index_data='^GSPC'
)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  50 of 50 completed
[*********************100%%**********************]  4 of 4 completed


## Dump data from the data dock to qlib database format

In [16]:
""" dump_mode: str
    "dump_all": Creates Qdata from zero. If a data for the same company already exists it will be deleted.
    "dump_update": Updates the data on existing dataset. Existing values wont be changed.
    "dump_fix": Replaces values on existing dataset.
"""
dump_mode = "dump_all"

# -------------------------------------------------------------------------------------------------------------

# Dump Data as Qlib database
cmd_command = (
    sys.executable + " " + str(Path().resolve().joinpath("scripts")) + "/dump_bin.py " + dump_mode
    + " --csv_path " + str(qlib_dir.joinpath("_datadock"))
    + " --qlib_dir " + str(qlib_dir)
    + " --date_field_name date"
    + " --include_fields open,close,high,low,volume,open_idx,high_idx,low_idx,close_idx,volume_idx"
)
os.system(cmd_command)

0